embedding 模型启动命令：

vllm serve Qwen/Qwen3-Embedding-0.6B --task embedding --host 0.0.0.0 --port 5001 --served-model-name custom_embedding

LLM 启动命令：

vllm serve Qwen/Qwen2.5-7B-Instruct --host 0.0.0.0 --port 5002 --served-model-name qwen25 --tensor-parallel-size 1 --gpu-memory-utilization 0.9

In [ ]:
# 定义模型对话调用函数
import requests
def call_model(messages, temperature=0.7):
    endpoint = 'http://localhost:5002/v1/chat/completions'
    body = {
        "model": "qwen25",
        'messages': messages,
        'temperature': temperature,
    }
    response = requests.post(endpoint, json=body)
    return response.json()['choices'][0]['message']['content']

In [ ]:
# 记忆对话demo
from mem0 import Memory

config = {
    "embedder": {
        "provider": "openai",        # 关键：走 langchain provider
        "config": {
            "api_key": "EMPTY",
            "openai_base_url": "http://localhost:5001/v1",
            "model": "custom_embedding",
        },
    },
    # 向量库配置
    "vector_store": {
        "provider": "upstash_vector",
        "config": {
            "url": "xxx",  # TODO: 注意替换，upstash创建索引时会给
            "token": "xxx",
            "collection_name": "memory_test",  # upstash 的 namespace
            "enable_embeddings": False,  #  是否使用upstash内置的embedding模型
        }
    },
    "llm": {
        "provider": "vllm",
        "config": {
            "model": "qwen25",  # 对应你网关里用的 model 名
            # 注意：这里是 base_url，不要带 /chat/completions
            "vllm_base_url": "http://localhost:5002/v1",
            "api_key": "EMPTY",
        },
    },
}

memory = Memory.from_config(config)
conversation_history = []

def chat_with_memories(user_input: str, user_id: str = "default_user") -> str:
    # 1. 对话前先去记忆库中检索是否有需要用到的记忆
    relevant_memories = memory.search(query=user_input, user_id=user_id, limit=3)
    memories_str = "\n".join(f"- {entry['memory']}" for entry in relevant_memories["results"])
    system_prompt = f"You are a helpful AI. Answer the question based on query and memories.\nUser Memories:\n{memories_str}"

    # 2. 把历史对话和当前输入一起传给 LLM，生成回复
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(conversation_history)  # 之前的 user/assistant 轮次
    messages.append({"role": "user", "content": user_input})

    assistant_response = call_model(messages=messages)

    conversation_history.append({"role": "user", "content": user_input})
    conversation_history.append({"role": "assistant", "content": assistant_response})

    # 3. 基于之前的对话生成或更新记忆库
    memory.add(messages, user_id=user_id)
    return assistant_response

def main():
    print("Chat with AI (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        res = chat_with_memories(user_input)
        print(f"AI: {res}")

main()

In [ ]:
# 查看用户已有的所有记忆
memory.get_all(user_id="default_user")

# 离线添加单条记忆
memory.add("我喜欢吃牛排", user_id="default_user", metadata={"category": "美食爱好"})

# 查询特定id的记忆
memory.get("eba57c92-37b9-402e-8d57-202421babec6")

# 搜索相关记忆
memory.search("我喜欢吃什么？", user_id="default_user")

# 修改指定记忆
memory.update(memory_id="ecd141ca-c8ac-42b3-801e-b86d6faf1d9e", data="喜欢吃炸鸡")

# 删除指定记忆
memory.delete("292aa7ad-c697-4eb0-9046-8aff730261cb")

# 删除某用户的所有记忆
memory.delete_all(user_id="default_user")

# 删除所有用户的所有记忆
memory.reset()